# Web Scraping Exercise

Web Scraping allows you to gather large volumes of data from diverse and real-time online sources. This data can be crucial for enriching your datasets, filling in gaps, and providing current information that enhances the quality and relevance of your analysis. Web scraping enables you to collect data that might not be readily available through traditional APIs or databases, offering a competitive edge by incorporating unique and comprehensive insights. Moreover, it automates the data collection process, saving time and resources while ensuring a scalable approach to continuously updating and maintaining your datasets.

Ethical web scraping involves respecting website terms of service, avoiding overloading servers, and ensuring that the collected data is used responsibly and in compliance with privacy laws and regulations.

Use Python, ```requests```, ```BeautifulSoup``` and/or ```pandas``` to scrape web data:

## Import Libraries

In [31]:
from pathlib import Path
import requests
import pandas as pd
from bs4 import BeautifulSoup

## Define the Target URL

In [32]:
url = "https://news.google.com/rss/search?q=Tesla%20stock&hl=en-US&gl=US&ceid=US:en"

## Send a Request to the Website

Do not forget to check the response status code

In [33]:
headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers, timeout=10)

print("Status Code:", response.status_code)

if response.status_code == 200:
    print("Request successful.")
else:
    print("Request failed.")

Status Code: 200
Request successful.


## Parse the HTML Content

Use a library to access the HTMl content

In [34]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(response.content, "lxml-xml")
items = soup.find_all("item")

print(len(items))

101


## Identify the Data to be Scraped

Write a couple of sentence on the data you want to scrape

We are scraping recent Tesla stock-related news from the Google News RSS feed.
The dataset contains structured information including article titles, publication dates, source names, and article links.
This data can be used for further analysis, such as identifying news trends or comparing news activity with Tesla stock price movements over time.

## Extract Data

Find specific elements and extract text or attributes from elements (handle pagination if necessary)

In [35]:
news_data = []

for item in items:
    title = item.find("title").text if item.find("title") else None
    link = item.find("link").text if item.find("link") else None
    pub_date = item.find("pubDate").text if item.find("pubDate") else None
    source = item.find("source").text if item.find("source") else None
    
    news_data.append({
        "title": title,
        "published_at": pub_date,
        "source": source,
        "link": link
    })

scraped_df = pd.DataFrame(news_data)

scraped_df.head()

,title,published_at,source,link
0,SpaceX IPO Adds Second Musk Stock. It’s a Prob...,"Tue, 19 May 2026 07:24:10 GMT",Yahoo Finance,https://news.google.com/rss/articles/CBMilwFBV...
1,SpaceX IPO Will Add Another Musk Stock. It’s a...,"Mon, 18 May 2026 17:37:45 GMT",Bloomberg.com,https://news.google.com/rss/articles/CBMitAFBV...
2,Why Tesla stock is falling: SpaceX IPO threate...,"Mon, 18 May 2026 19:47:11 GMT",Investing.com,https://news.google.com/rss/articles/CBMixAFBV...
3,"Tesla Hikes Prices, and the Stock Takes a Hit ...","Tue, 19 May 2026 12:50:00 GMT",Barron's,https://news.google.com/rss/articles/CBMiggFBV...
4,"This EV Stock Could Soar By 79%, According to ...","Tue, 19 May 2026 10:30:00 GMT",The Motley Fool,https://news.google.com/rss/articles/CBMimAFBV...


## Store Data in a Structured Format

Give a brief overview of the data collected (e.g. count, fields, ...)

In [36]:
scraped_df["published_at"] = pd.to_datetime(scraped_df["published_at"], errors="coerce", utc=True)
scraped_df["date"] = scraped_df["published_at"].dt.date

scraped_df = scraped_df.dropna(subset=["title", "published_at"])
scraped_df = scraped_df.drop_duplicates(subset=["title"])
scraped_df = scraped_df.sort_values("published_at", ascending=False)

print("Number of Tesla news articles:", len(scraped_df))
print("Columns:", list(scraped_df.columns))

scraped_df.head()

Number of Tesla news articles: 84
Columns: ['title', 'published_at', 'source', 'link', 'date']


,title,published_at,source,link,date
27,Institutional buying lifts Tesla stock 2.17% h...,2026-05-20 15:15:56+00:00,Traders Union,https://news.google.com/rss/articles/CBMimgFBV...,2026-05-20
7,Want to invest in the AI robotics revolution? ...,2026-05-20 14:46:20+00:00,Yahoo Finance UK,https://news.google.com/rss/articles/CBMiiwFBV...,2026-05-20
19,"Tesla Ends Model S and X Era, Bets Everything ...",2026-05-20 14:15:46+00:00,MarketBeat,https://news.google.com/rss/articles/CBMinAFBV...,2026-05-20
81,Tesla stock (US88160R1014): Robotaxi rollout a...,2026-05-20 14:11:47+00:00,AD HOC NEWS,https://news.google.com/rss/articles/CBMixgFBV...,2026-05-20
20,Tesla Stock Outlook: Can TSLA Stock Climb Back...,2026-05-20 14:06:42+00:00,TradingKey,https://news.google.com/rss/articles/CBMi5wFBV...,2026-05-20


## Save the Data

In [37]:
project_root = Path.cwd().parent

processed_dir = project_root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

output_path = processed_dir / "tesla_news_cleaned.csv"

scraped_df.to_csv(output_path, index=False)

print("Saved file to:", output_path)

Saved file to: /home/admin/Dokumente/A_4_Semester_BWI/Big-Data-Engineering/data/processed/tesla_news_cleaned.csv


## Check if file exists

In [38]:
check_path = processed_dir / "tesla_news_cleaned.csv"

print("File exists:", check_path.exists())

check_df = pd.read_csv(check_path)
print("Rows:", len(check_df))
check_df.head()

File exists: True
Rows: 84


,title,published_at,source,link,date
0,Institutional buying lifts Tesla stock 2.17% h...,2026-05-20 15:15:56+00:00,Traders Union,https://news.google.com/rss/articles/CBMimgFBV...,2026-05-20
1,Want to invest in the AI robotics revolution? ...,2026-05-20 14:46:20+00:00,Yahoo Finance UK,https://news.google.com/rss/articles/CBMiiwFBV...,2026-05-20
2,"Tesla Ends Model S and X Era, Bets Everything ...",2026-05-20 14:15:46+00:00,MarketBeat,https://news.google.com/rss/articles/CBMinAFBV...,2026-05-20
3,Tesla stock (US88160R1014): Robotaxi rollout a...,2026-05-20 14:11:47+00:00,AD HOC NEWS,https://news.google.com/rss/articles/CBMixgFBV...,2026-05-20
4,Tesla Stock Outlook: Can TSLA Stock Climb Back...,2026-05-20 14:06:42+00:00,TradingKey,https://news.google.com/rss/articles/CBMi5wFBV...,2026-05-20
